# Phase 2: Chunking Strategy Experiments

**Pipeline**: Vietnamese Financial News RAG System — v3  
**Sections covered**: §2 — fixed_size, sentence_aware, article_level  

**How to run**:
- **Local**: `jupyter notebook` from `implementation/`
- **Colab**: Mount Drive first (Cell 0), then run top-to-bottom

All cells are idempotent — safe to re-run without recomputing completed work.

## Cell 0 — Environment Setup

In [1]:
import os, sys
from pathlib import Path

# Detect environment
def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/rag-vn-finance/implementation')
    if not REPO_ROOT.exists():
        os.system('git clone https://github.com/thong7d/rag-vn-finance /content/rag-vn-finance')
else:
    REPO_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())

print(f"Project root: {REPO_ROOT}")
assert REPO_ROOT.exists(), f"Project root not found: {REPO_ROOT}"

src_path = str(REPO_ROOT)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

if IN_COLAB:
    req_path = REPO_ROOT / 'requirements.txt'
    if req_path.exists():
        os.system(f'pip install -r {req_path} -q')

print("\nCell 0 complete.")

Environment: Local
Project root: d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\implementation

Cell 0 complete.


## Cell 1 — Load Config & Cleaned Data

In [2]:
import pandas as pd
import json
from src.utils import load_config, resolve_path

config = load_config(REPO_ROOT / 'configs' / 'config.yaml')
print("Config loaded.")

processed_path = resolve_path(config['data'], 'processed_path')
if not os.path.isabs(processed_path):
    processed_path = str(REPO_ROOT / processed_path)

print(f"Loading cleaned data from: {processed_path}")
df = pd.read_parquet(processed_path)

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Verify Semantic Enrichment columns exist
required_meta = ['tickers', 'is_historical', 'numerical_density', 'entities']
for col in required_meta:
    assert col in df.columns, f"Missing Semantic Enrichment column: {col}"
print(f"\n✅ All Semantic Enrichment columns present.")
print(f"Sample doc_id: {df['doc_id'].iloc[0]}")
df.head(2)

Config loaded.
Loading cleaned data from: d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\implementation\data\processed\cleaned.parquet
Shape: (9999, 30)
Columns: ['url', 'title', 'time', 'category', 'content', 'tags', 'content_token_counts', 'year', 'month', 'yearmonth', 'doc_id', 'source', 'tickers', 'is_historical', 'numerical_density', 'entities', 'token_count', 'word_count', 'char_count', 'ttr', 'has_lãi_suất', 'has_chứng_khoán', 'has_cổ_phiếu', 'has_ngân_hàng', 'has_tín_dụng', 'has_đầu_tư', 'has_GDP', 'has_lạm_phát', 'has_tỷ_giá', 'has_vốn_hóa']

✅ All Semantic Enrichment columns present.
Sample doc_id: b2eda821f172bbee


,url,title,time,category,content,tags,content_token_counts,year,month,yearmonth,...,has_lãi_suất,has_chứng_khoán,has_cổ_phiếu,has_ngân_hàng,has_tín_dụng,has_đầu_tư,has_GDP,has_lạm_phát,has_tỷ_giá,has_vốn_hóa
0,https://vneconomy.vn/hang-ve-tranh-ban-thi-tru...,"Hàng về tranh bán, thị trường chìm trong sắc đ...",2023-03-03,Chứng khoán,"Biên độ tăng 15,87 điểm hôm 1/3 đã bị quét sạc...",nhận định chứng khoán,919,2023,3,2023-03,...,False,True,True,True,False,True,False,False,False,False
1,https://vneconomy.vn/bo-tai-chinh-lay-y-kien-d...,Bộ Tài chính lấy ý kiến đề xuất ưu đãi thuế th...,2024-06-13,Tài chính,"Theo Bộ Tài chính, mục tiêu đề xuất ưu đãi thu...","Bộ Tài Chính, doanh nghiệp, doanh nghiệp khoa ...",892,2024,6,2024-06,...,False,False,False,False,False,True,False,False,False,False


## Cell 2 — Chunking Configuration

In [3]:
chunk_cfg = config['chunking']
strategies = chunk_cfg['strategies']

chunks_base_dir = resolve_path(chunk_cfg, 'output_dir')
if not os.path.isabs(chunks_base_dir):
    chunks_base_dir = str(REPO_ROOT / chunks_base_dir)

print(f"Strategies: {strategies}")
print(f"Chunks output dir: {chunks_base_dir}")
print(f"\nfixed_size config:     {chunk_cfg['fixed_size']}")
print(f"sentence_aware config: {chunk_cfg['sentence_aware']}")
print(f"article_level config:  {chunk_cfg['article_level']}")

print("\nCell 2 complete.")

Strategies: ['fixed_size', 'sentence_aware', 'article_level']
Chunks output dir: d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\implementation\data\chunks

fixed_size config:     {'chunk_size': 256, 'overlap': 32}
sentence_aware config: {'max_sentences': 5, 'overlap_sentences': 1}
article_level config:  {'max_tokens': 512}

Cell 2 complete.


## Cell 3 — Strategy 1: Fixed-Size Chunking (256 tokens, 32 overlap)

> Each chunk is prefixed with `"Title: {title}\n"`.

In [4]:
from src.chunking import chunk_fixed_size, save_chunks, save_stats, generate_chunk_stats

fs_dir = os.path.join(chunks_base_dir, 'fixed_size')
fs_parquet = os.path.join(fs_dir, 'chunks.parquet')

if not os.path.exists(fs_parquet):
    print("Running fixed_size chunking...")
    df_fs = chunk_fixed_size(
        df,
        chunk_size=chunk_cfg['fixed_size']['chunk_size'],
        overlap=chunk_cfg['fixed_size']['overlap'],
    )
    save_chunks(df_fs, fs_dir)
    fs_stats = generate_chunk_stats(df_fs, 'fixed_size')
    save_stats(fs_stats, fs_dir)
else:
    print(f"Cached: {fs_parquet}")
    df_fs = pd.read_parquet(fs_parquet)
    fs_stats = json.load(open(os.path.join(fs_dir, 'stats.json')))

print(f"\nfixed_size chunks: {len(df_fs):,}")
print(f"Stats: {json.dumps(fs_stats, indent=2)}")
print(f"\nSample chunk:")
display(df_fs.head(1))

Cached: d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\implementation\data\chunks\fixed_size\chunks.parquet

fixed_size chunks: 45,764
Stats: {
  "strategy": "fixed_size",
  "total_chunks": 45764,
  "total_articles": 9999,
  "avg_tokens_per_chunk": 250.9,
  "max_tokens_per_chunk": 299,
  "min_tokens_per_chunk": 43,
  "truncated_chunks": 36678,
  "chunks_per_article": {
    "mean": 4.6,
    "median": 5.0,
    "max": 9
  }
}

Sample chunk:


,chunk_id,doc_id,text,chunk_index,total_chunks,strategy,source,category,time,year,title,url,tickers,is_historical,numerical_density,entities
0,b2eda821f172bbee_c0000,b2eda821f172bbee,"Title: Hàng về tranh bán, thị trường chìm tron...",0,4,fixed_size,vneconomy.vn,Chứng khoán,2023-03-03T00:00:00,2023,"Hàng về tranh bán, thị trường chìm trong sắc đ...",https://vneconomy.vn/hang-ve-tranh-ban-thi-tru...,[],False,0.070348,"1/3,cuối,2,hôm,Khoảng,Kết phiên,15,87 điểm,T,s..."


## Cell 4 — Strategy 2: Sentence-Aware Chunking (5 sentences, 1 overlap)

> Fallback: if one sentence > 256 tokens, it is split by token.

In [5]:
from src.chunking import chunk_sentence_aware, save_chunks, save_stats, generate_chunk_stats

sa_dir = os.path.join(chunks_base_dir, 'sentence_aware')
sa_parquet = os.path.join(sa_dir, 'chunks.parquet')

if not os.path.exists(sa_parquet):
    print("Running sentence_aware chunking...")
    df_sa = chunk_sentence_aware(
        df,
        max_sentences=chunk_cfg['sentence_aware']['max_sentences'],
        overlap_sentences=chunk_cfg['sentence_aware']['overlap_sentences'],
    )
    save_chunks(df_sa, sa_dir)
    sa_stats = generate_chunk_stats(df_sa, 'sentence_aware')
    save_stats(sa_stats, sa_dir)
else:
    print(f"Cached: {sa_parquet}")
    df_sa = pd.read_parquet(sa_parquet)
    sa_stats = json.load(open(os.path.join(sa_dir, 'stats.json')))

print(f"\nsentence_aware chunks: {len(df_sa):,}")
print(f"Stats: {json.dumps(sa_stats, indent=2)}")
print(f"\nSample chunk:")
display(df_sa.head(1))

Cached: d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\implementation\data\chunks\sentence_aware\chunks.parquet

sentence_aware chunks: 64,197
Stats: {
  "strategy": "sentence_aware",
  "total_chunks": 64197,
  "total_articles": 9999,
  "avg_tokens_per_chunk": 186.1,
  "max_tokens_per_chunk": 500,
  "min_tokens_per_chunk": 3,
  "truncated_chunks": 0,
  "chunks_per_article": {
    "mean": 6.4,
    "median": 6.0,
    "max": 19
  }
}

Sample chunk:


,chunk_id,doc_id,text,chunk_index,total_chunks,strategy,source,category,time,year,title,url,tickers,is_historical,numerical_density,entities
0,b2eda821f172bbee_c0000,b2eda821f172bbee,"Biên độ tăng 15,87 điểm hôm 1/3 đã bị quét sạc...",0,10,sentence_aware,vneconomy.vn,Chứng khoán,2023-03-03T00:00:00,2023,"Hàng về tranh bán, thị trường chìm trong sắc đ...",https://vneconomy.vn/hang-ve-tranh-ban-thi-tru...,[],False,0.070348,"1/3,cuối,2,hôm,Khoảng,Kết phiên,15,87 điểm,T,s..."


## Cell 5 — Strategy 3: Article-Level Chunking (max 512 tokens)

> One chunk per article, truncated at 512 tokens.

In [6]:
from src.chunking import chunk_article_level

al_dir = os.path.join(chunks_base_dir, 'article_level')
al_parquet = os.path.join(al_dir, 'chunks.parquet')

if not os.path.exists(al_parquet):
    print("Running article_level chunking...")
    df_al = chunk_article_level(
        df,
        max_tokens=chunk_cfg['article_level']['max_tokens'],
    )
    save_chunks(df_al, al_dir)
    al_stats = generate_chunk_stats(df_al, 'article_level')
    save_stats(al_stats, al_dir)
else:
    print(f"Cached: {al_parquet}")
    df_al = pd.read_parquet(al_parquet)
    al_stats = json.load(open(os.path.join(al_dir, 'stats.json')))

print(f"\narticle_level chunks: {len(df_al):,}")
print(f"Stats: {json.dumps(al_stats, indent=2)}")
print(f"\nSample chunk:")
display(df_al.head(1))

Cached: d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\implementation\data\chunks\article_level\chunks.parquet

article_level chunks: 9,999
Stats: {
  "strategy": "article_level",
  "total_chunks": 9999,
  "total_articles": 9999,
  "avg_tokens_per_chunk": 494.3,
  "max_tokens_per_chunk": 512,
  "min_tokens_per_chunk": 77,
  "truncated_chunks": 8537,
  "chunks_per_article": {
    "mean": 1.0,
    "median": 1.0,
    "max": 1
  }
}

Sample chunk:


,chunk_id,doc_id,text,chunk_index,total_chunks,strategy,source,category,time,year,title,url,tickers,is_historical,numerical_density,entities
0,b2eda821f172bbee_c0000,b2eda821f172bbee,"Biên độ tăng 15,87 điểm hôm 1/3 đã bị quét sạc...",0,1,article_level,vneconomy.vn,Chứng khoán,2023-03-03T00:00:00,2023,"Hàng về tranh bán, thị trường chìm trong sắc đ...",https://vneconomy.vn/hang-ve-tranh-ban-thi-tru...,[],False,0.070348,"1/3,cuối,2,hôm,Khoảng,Kết phiên,15,87 điểm,T,s..."


## Cell 6 — Strategy Comparison Summary

In [7]:
summary = pd.DataFrame([
    {'strategy': s['strategy'], 'total_chunks': s['total_chunks'],
     'avg_tokens': s['avg_tokens_per_chunk'], 'max_tokens': s['max_tokens_per_chunk'],
     'truncated': s['truncated_chunks'],
     'chunks_per_article_mean': s['chunks_per_article']['mean']}
    for s in [fs_stats, sa_stats, al_stats]
])

print("=" * 60)
print("CHUNKING STRATEGY COMPARISON")
print("=" * 60)
display(summary)

# Verify metadata propagation
meta_cols = ['source', 'category', 'time', 'year', 'title', 'url',
             'tickers', 'is_historical', 'numerical_density', 'entities']
for name, df_c in [('fixed_size', df_fs), ('sentence_aware', df_sa), ('article_level', df_al)]:
    missing = [c for c in meta_cols if c not in df_c.columns]
    if missing:
        print(f"⚠️  {name} missing metadata: {missing}")
    else:
        print(f"✅ {name} — all metadata columns present")

print("\n✅ Phase 2 complete. Confirm 'Xong' before proceeding to Phase 3 (Embedding).")

CHUNKING STRATEGY COMPARISON


,strategy,total_chunks,avg_tokens,max_tokens,truncated,chunks_per_article_mean
0,fixed_size,45764,250.9,299,36678,4.6
1,sentence_aware,64197,186.1,500,0,6.4
2,article_level,9999,494.3,512,8537,1.0


✅ fixed_size — all metadata columns present
✅ sentence_aware — all metadata columns present
✅ article_level — all metadata columns present

✅ Phase 2 complete. Confirm 'Xong' before proceeding to Phase 3 (Embedding).
